# Nutrition AI Agent
**Team Dietify tech** — Nutrition & Diet Assistant

This notebook implements a working first version of the AI Agent described in our
Day 1 pitch: a conversational agent that logs a user's meals in plain language,
checks them against that user's **goals, allergies, diet type, and budget**, and
gives an instant, personalized recommendation — instead of a generic diet plan or
a $ dietitian consultation.

### Agent design (matches the pitch)
| | |
|---|---|
| **Input** | A short text description of what the user ate/wants to eat, plus a one-time user profile (goals, allergies, diet type, budget tier) |
| **Perceive** | Match the free-text meal description against a nutrition database |
| **Decide** | Check the matched food against allergens, daily calorie/macro targets, and budget fit |
| **Tools** | `lookup_food()`, `check_allergens()`, `check_diet_fit()`, `check_budget_fit()`, `suggest_alternative()`, optional Claude API call for natural-language phrasing |
| **Output/Action** | A personalized message: ✅ good fit, ⚠️ warning (e.g. allergen or over-budget on calories), or a swapped-in alternative food |

This is intentionally an **MVP**: the food database is a small CSV sample
(`data/sample_food_data.csv`) standing in for USDA FoodData Central / Edamam,
which is what we costed out in the pitch deck. The agent's control flow (parse →
check → decide → respond) is the same logic that would sit in front of a
production-scale food database and LLM.

## 1. Setup

In [ ]:
# If running in Colab, uncomment:
# !pip install pandas anthropic -q

import pandas as pd
import os
import re

pd.set_option('display.max_colwidth', None)
print("Environment ready.")

## 2. Load the nutrition database (a tool the agent relies on)

In [ ]:
# Works both locally (data/sample_food_data.csv) and on Google Colab,
# where local repo files aren't present by default -- falls back to
# pulling the CSV straight from GitHub.
import os

local_path = "data/sample_food_data.csv"
github_raw_url = "https://raw.githubusercontent.com/omchoudhari431/Dietify-tech/main/data/sample_food_data.csv"

if os.path.exists(local_path):
    food_db = pd.read_csv(local_path)
else:
    food_db = pd.read_csv(github_raw_url)

food_db["allergens"] = food_db["allergens"].fillna("none")
food_db["diet_tags"] = food_db["diet_tags"].fillna("")
food_db.head(10)

## 3. Agent tools

Each function below is a discrete "tool" the agent calls when reasoning about a meal —
mirroring the tool list from our problem statement (food lookup, allergy check, budget/goal check).

In [ ]:
def lookup_food(text, db=food_db):
    """Perception tool: match free-text meal description to a row in the food database."""
    text_lower = text.lower()
    matches = db[db["food_name"].apply(lambda name: name.lower() in text_lower)]
    if matches.empty:
        # fall back to loose word-overlap match
        words = set(re.findall(r"[a-z]+", text_lower))
        def overlap(name):
            return len(words & set(name.lower().split()))
        db = db.copy()
        db["score"] = db["food_name"].apply(overlap)
        matches = db[db["score"] > 0].sort_values("score", ascending=False)
    return matches.iloc[0] if not matches.empty else None


def check_allergens(food_row, user_allergies):
    """Decision tool: does this food contain something the user is allergic to?"""
    food_allergens = set(food_row["allergens"].split(";")) if food_row["allergens"] != "none" else set()
    hit = food_allergens & set(user_allergies)
    return list(hit)  # empty list = safe


def check_diet_fit(food_row, diet_type):
    """Decision tool: does this food match the user's diet type (vegan/vegetarian/none)?"""
    if diet_type == "none":
        return True
    tags = food_row["diet_tags"].split(";") if food_row["diet_tags"] else []
    return diet_type in tags


def check_budget_fit(food_row, budget_tier):
    """Decision tool: is this food within the user's budget tier?"""
    tiers = {"low": 1, "medium": 2, "high": 3}
    return tiers.get(food_row["price_tier"], 2) <= tiers.get(budget_tier, 2)


def suggest_alternative(food_row, user_profile, db=food_db):
    """Action tool: find a safer/better-fit alternative with similar calories."""
    candidates = db.copy()
    candidates = candidates[candidates["food_name"] != food_row["food_name"]]
    candidates = candidates[candidates.apply(
        lambda r: not check_allergens(r, user_profile["allergies"]), axis=1)]
    if user_profile["diet_type"] != "none":
        candidates = candidates[candidates.apply(
            lambda r: check_diet_fit(r, user_profile["diet_type"]), axis=1)]
    candidates = candidates[candidates.apply(
        lambda r: check_budget_fit(r, user_profile["budget_tier"]), axis=1)]
    if candidates.empty:
        return None
    candidates["cal_diff"] = (candidates["calories"] - food_row["calories"]).abs()
    return candidates.sort_values("cal_diff").iloc[0]

## 4. The Agent

Ties the tools together: perceive the meal → run every check → decide an action → respond.

In [ ]:
class NutritionAgent:
    def __init__(self, user_profile, db=food_db):
        """
        user_profile example:
        {
            "name": "Priya",
            "daily_calorie_goal": 1800,
            "allergies": ["dairy", "peanuts"],
            "diet_type": "vegetarian",   # vegan / vegetarian / none
            "budget_tier": "low"          # low / medium / high
        }
        """
        self.profile = user_profile
        self.db = db
        self.calories_logged_today = 0

    def log_meal(self, meal_text):
        food_row = lookup_food(meal_text, self.db)
        if food_row is None:
            return {
                "status": "not_found",
                "message": f"I couldn't find \"{meal_text}\" in the food database yet. "
                           f"Try describing it differently, or add it to data/sample_food_data.csv."
            }

        allergy_hits = check_allergens(food_row, self.profile["allergies"])
        diet_ok = check_diet_fit(food_row, self.profile["diet_type"])
        budget_ok = check_budget_fit(food_row, self.profile["budget_tier"])
        projected_calories = self.calories_logged_today + food_row["calories"]
        over_goal = projected_calories > self.profile["daily_calorie_goal"]

        result = {
            "matched_food": food_row["food_name"],
            "calories": int(food_row["calories"]),
            "protein_g": food_row["protein_g"],
            "allergy_hits": allergy_hits,
            "diet_ok": diet_ok,
            "budget_ok": budget_ok,
            "over_calorie_goal": over_goal,
        }

        # Decision logic
        if allergy_hits:
            result["status"] = "blocked_allergen"
            alt = suggest_alternative(food_row, self.profile, self.db)
            result["alternative"] = alt["food_name"] if alt is not None else None
        elif not diet_ok:
            result["status"] = "blocked_diet"
            alt = suggest_alternative(food_row, self.profile, self.db)
            result["alternative"] = alt["food_name"] if alt is not None else None
        elif not budget_ok:
            result["status"] = "over_budget"
            alt = suggest_alternative(food_row, self.profile, self.db)
            result["alternative"] = alt["food_name"] if alt is not None else None
        elif over_goal:
            result["status"] = "over_calorie_goal"
        else:
            result["status"] = "ok"
            self.calories_logged_today = projected_calories

        result["message"] = self._render_message(result)
        return result

    def _render_message(self, r):
        """Fallback rule-based natural-language response (no API key required).
        See section 5 for an optional Claude-powered version."""
        food = r["matched_food"]
        if r["status"] == "blocked_allergen":
            alt_txt = f" Try **{r['alternative']}** instead." if r["alternative"] else ""
            return f"⚠️ **{food}** contains an allergen you listed ({', '.join(r['allergy_hits'])}) — I'd skip it.{alt_txt}"
        if r["status"] == "blocked_diet":
            alt_txt = f" **{r['alternative']}** fits your diet and is a similar calorie level." if r["alternative"] else ""
            return f"⚠️ **{food}** doesn't match your {self.profile['diet_type']} diet.{alt_txt}"
        if r["status"] == "over_budget":
            alt_txt = f" **{r['alternative']}** is a cheaper option with similar calories." if r["alternative"] else ""
            return f"💸 **{food}** is above your usual budget tier.{alt_txt}"
        if r["status"] == "over_calorie_goal":
            return (f"📊 **{food}** ({r['calories']} kcal) would put you over your "
                    f"{self.profile['daily_calorie_goal']} kcal goal for today. Consider a lighter option "
                    f"or adjust portion size.")
        return f"✅ **{food}** ({r['calories']} kcal, {r['protein_g']}g protein) logged — good fit for your goals!"

## 5. Optional: Claude-powered natural-language layer

The rule-based `_render_message` above already makes the agent fully functional offline.
This cell shows how the same decision output (`result` dict) can be handed to the
**Claude API** to produce a warmer, more conversational reply — the "LLM engine" line item
from our cost breakdown slide. It's optional: if no API key is set, the notebook keeps
using the rule-based messages above so the agent still runs end-to-end for demo/grading.

In [ ]:
USE_CLAUDE = bool(os.environ.get("ANTHROPIC_API_KEY"))

if USE_CLAUDE:
    import anthropic
    client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment

    def render_with_claude(decision_result, user_profile):
        prompt = (
            f"You are a friendly nutrition assistant. A user named {user_profile['name']} "
            f"logged this meal check result: {decision_result}. "
            f"In 1-2 short sentences, give them warm, encouraging, plain-language feedback. "
            f"Do not give medical advice."
        )
        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=150,
            messages=[{"role": "user", "content": prompt}],
        )
        return response.content[0].text
else:
    print("ANTHROPIC_API_KEY not set — using the rule-based responses from section 4. "
          "Set the environment variable and re-run this cell to enable Claude-generated replies.")

## 6. Demo run

In [ ]:
user_profile = {
    "name": "Priya",
    "daily_calorie_goal": 1800,
    "allergies": ["dairy", "peanuts"],
    "diet_type": "vegetarian",
    "budget_tier": "low",
}

agent = NutritionAgent(user_profile)

test_meals = [
    "I had paneer curry with roti for lunch",
    "grabbed a peanut butter toast for breakfast",
    "had a burger for dinner",
    "ate some dal and rice",
    "drank a protein shake after the gym",
]

for meal in test_meals:
    result = agent.log_meal(meal)
    print(f"You said: \"{meal}\"")
    if USE_CLAUDE and result.get("status") != "not_found":
        print(render_with_claude(result, user_profile))
    else:
        print(result["message"])
    print("-" * 70)

print(f"\nTotal calories logged today: {agent.calories_logged_today} / {user_profile['daily_calorie_goal']} kcal")

## 7. Limitations & next steps (honest debugging notes)

- **Food matching** is keyword-based, not NLP/embedding-based — it won't handle typos or
  unusual phrasing well. Next step: swap `lookup_food()` for a Claude tool-call that extracts
  structured food items from free text, or embeddings-based fuzzy search over a full
  USDA FoodData Central dump.
- **Database** is a 26-row sample. Production version wires in USDA FoodData Central (free)
  or Edamam (paid tier at scale), as costed in our pitch.
- **No persistence** yet — `calories_logged_today` resets every notebook run. Next step:
  a small SQLite/Firestore layer per user.
- **No photo input** yet, though the pitch highlights "photo or text" logging — next step:
  send a food photo to a vision-capable Claude call and parse the identified dish back into
  `lookup_food()`.
